# Own NDT network on KITTI, trained with the BEVPlace++ recipe

Runtime -> Change runtime type -> **GPU** (A100 / L4 recommended: 8 rotations x 2000 cells per cloud).
Set cell 1, then **Runtime -> Run all**. `MODE` works as in notebook 01 (train / eval / smoke).

- **Network** (`src/lpr/models/ndt_net.py`, own implementation): NDT cells -> [T-Net] -> conv1 -> Transformer -> conv2 -> NetVLAD.
  `rotations=8`: REIN-like, as BEVPlace++ — the cells are rotated by 0, 45, ..., 315 deg, every copy goes through
  the same encoder and each cell keeps the maximum over the copies. `head`: `vlad` (NetVLAD of BEVPlace++, 8192-d)
  or `loupe` (NetVLADLoupe of NDT-Transformer, 256-d). `stn`: the T-Net of NDT-Transformer (off by default).
- **Recipe**: the training code of the BEV runs (`src/lpr/bevplace.py`): triplet loss margin 0.3 on the hardest of
  10 negatives, hard mining from a descriptor cache of all training frames every epoch, batch 4, Adam 1e-4, 40 epochs,
  random rotation of every training sample. Only the network and its input differ from BEVPlace++.
- **NDT variant**: the same variants as notebook 01 (`MyDrive/lpr/ndt/<variant>/`).
- **Run**: `MyDrive/lpr/runs/myndt/<run>/` (`config.json` with every parameter, `metrics.csv`, a checkpoint per epoch);
  evaluations in `<run>/eval/<type>/<checkpoint>/`, in the same tables as the other runs (`runs_summary.csv`, `evals_summary.csv`, method `myndt`).


## 1. Settings

In [ ]:
REPO_URL = "https://github.com/dov85/lidar-place-recognition"
DRIVE = "/content/drive/MyDrive/lpr"
SEQ = "00"                          # training sequence
# What this run of the notebook does. "Runtime -> Run all" works in every mode:
#   "train": data, training (a new run, or the continuation of an unfinished one; a finished run
#            is not trained again), then the evaluations
#   "eval":  only the evaluations of an existing run (nothing is trained or built)
#   "smoke": a short training test (time and GPU memory), nothing is saved to Drive
MODE = "train"
EVAL_CKPT = "last"                  # "last" or a checkpoint file of the run, e.g. "epoch_18.pth"
EVALS = ['seqs_heading', 'compass']   # types in src/lpr/evals.py (EVAL_TYPES); tnet is for notebook 01

# NDT variant: only what differs from the defaults (src/lpr/experiments.py, NDT_DEFAULTS):
#   crop="square", half_size=40.0, ground="patchwork", k=2000, over=0.10, radius=0.8,
#   min_pts=5, knn=8, cov_eps=1e-3
NDT_TAG = ""          # name for a changed variant; empty = date + hash
NDT_PARAMS = dict()                 # e.g. dict(crop="circle", radius=1.0)

# Training run: only what differs from the defaults (MYNDT_TRAIN_DEFAULTS in src/lpr/experiments.py):
#   recipe (as BEVPlace++): train_frames=3000, pos_thr=5.0, neg_thr=7.0, epochs=40, lr=1e-4, batch=4,
#     negatives=10, margin=0.3, rot_aug=True, seed=1024,
#     rot_points=0  (N > 0: training clouds from the N point-rotated copies of the variant, built on Azure),
#     amp=True      (mixed precision)
#   network: stn=False (T-Net), rotations=8 (1 = no rotations), head="vlad" ("loupe"), feat_dim=128,
#     cov_transform=False (only with stn=True)
RUN_TAG = ""          # name for a changed run; empty = date + hash
TRAIN_PARAMS = dict()               # e.g. dict(rotations=1), dict(head="loupe"), dict(stn=True, rotations=1)

KITTI = "/content/kitti"                       # poses (+ raw scans when a variant is built here)
RAW_TAR = f"{DRIVE}/kitti_seq00_raw.tar"     # backup of the raw scans of sequence 00

## 2. Drive, GPU, code, poses, Drive layout

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, sys, json, subprocess, importlib, glob
assert MODE in ("train", "eval", "smoke"), MODE
os.makedirs(DRIVE, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

url = REPO_URL
try:  # private repo: store a token in Colab Secrets as GITHUB_TOKEN
    url = REPO_URL.replace("https://", f"https://{userdata.get('GITHUB_TOKEN')}@")
except Exception:
    pass
if not os.path.exists("/content/lpr"):
    subprocess.run(["git", "clone", "-q", url, "/content/lpr"], check=True)
else:
    subprocess.run(["git", "-C", "/content/lpr", "pull", "-q"], check=True)
%cd /content/lpr
!git log --oneline -1
!bash scripts/setup_third_party.sh
!pip install -q -r requirements.txt faiss-cpu   # faiss: k-means initialisation of NetVLAD

sys.path.insert(0, "/content/lpr/src")
import lpr.experiments as ex, lpr.evals as ev
importlib.reload(ex); importlib.reload(ev)
missing_poses = [s for s in ev.ALL_SEQS if not os.path.exists(f"{KITTI}/poses/{s}.txt")]
if missing_poses:
    !python scripts/download_kitti.py --out {KITTI} --seqs {" ".join(missing_poses)} --poses_only

# older results into the current Drive layout (does nothing once done)
if os.path.exists(f"{DRIVE}/runs/ndt_kitti00"):
    !python scripts/migrate_stage1.py --drive {DRIVE} --apply
!python scripts/migrate_layout.py --drive {DRIVE} --apply

## 3. Names of the NDT variant and the run

In [ ]:
VARIANT, NDT_FULL, ndt_exists = ex.resolve_ndt_variant(DRIVE, NDT_PARAMS, NDT_TAG)
VDIR = f"{DRIVE}/ndt/{VARIANT}"          # permanent copy on Drive
NDT = f"/content/ndt/{VARIANT}"          # working copy on this machine
CELLS_TAR = f"{VDIR}/cells_{SEQ}.tar"

RUN, TRAIN_FULL, run_state = ex.resolve_run(DRIVE, dict(TRAIN_PARAMS, seq=SEQ), VARIANT, RUN_TAG, method="myndt")
RUN_DIR = f"{DRIVE}/runs/myndt/{RUN}"
print("mode:       ", MODE)
print("NDT variant:", VARIANT, "(on Drive)" if ndt_exists else "(new)")
print(json.dumps(NDT_FULL))
print("run:        ", RUN, f"({run_state})")
print(json.dumps(TRAIN_FULL))
if MODE == "eval" and run_state == "new":
    raise SystemExit("MODE = 'eval', but there is no run with these parameters: check TRAIN_PARAMS / NDT_PARAMS")

t = TRAIN_FULL
flags = " ".join(f"--{k}" if t[k] else f"--no_{k}" for k in ("rot_aug", "amp", "stn", "cov_transform"))
train_args = (f"--ndt_dir {NDT} --kitti_root {KITTI} --ndt_variant {VARIANT} " +
              " ".join(f"--{k} {t[k]}" for k in ("seq", "train_frames", "pos_thr", "neg_thr", "epochs", "lr",
                                                  "batch", "negatives", "margin", "seed", "rot_points",
                                                  "rotations", "head", "feat_dim")) + " " + flags)
print(train_args)


## 4. NDT cells of the training sequence: restore from Drive, or build the variant
Building restores the raw scans from `RAW_TAR` (or downloads them) and runs preprocessing + G on every scan
(about 0.5 s per scan on 2 CPU cores). In `eval` mode nothing is built.
Every build also runs the built-in checks: coverage of the points by the cells (columns `cov_0.8`, `dense_cov`, ...
in `stats_<seq>.csv`) and pictures of 10 random scans in `<variant>/previews_<seq>/`; notebook 03 shows them.

With `rot_points=N` in `TRAIN_PARAMS` (train / smoke): the N point-rotated copies of the training frames
(`cells_<seq>_rot<N>.tar` of the variant) are copied from the Azure blob container to Drive, and from Drive
to this machine. If they are not built yet, this cell prints the command for the Azure VM and stops.

In [ ]:
from IPython.display import Image, display
ndt_args = " ".join(f"--{k} {v}" for k, v in NDT_FULL.items())

if os.path.exists(f"{NDT}/{SEQ}/000000.bin"):
    print("NDT cells already on this machine:", NDT)
elif os.path.exists(CELLS_TAR):
    os.makedirs(NDT, exist_ok=True)
    !tar -xf {CELLS_TAR} -C {NDT}
    !cp {VDIR}/config.json {NDT}/config.json
elif MODE == "eval":
    print(f"no cells of sequence {SEQ} on Drive (eval mode: not built)")
else:
    if not os.path.exists(f"{KITTI}/sequences/{SEQ}/velodyne"):
        if os.path.exists(RAW_TAR):
            !mkdir -p {KITTI} && tar -xf {RAW_TAR} -C {KITTI}
        else:
            !python scripts/download_kitti.py --out {KITTI} --seqs {SEQ} --threads 48
    os.makedirs(VDIR, exist_ok=True)
    os.makedirs(NDT, exist_ok=True)
    if os.path.exists(f"{VDIR}/config.json"):   # variant exists, new sequence: keep its history
        !cp {VDIR}/config.json {NDT}/config.json
    !python scripts/visualize_ndt.py --kitti_root {KITTI} --seq {SEQ} --idx 0 --variant {VARIANT} --out {VDIR}/preview_{SEQ}.png {ndt_args}
    display(Image(f"{VDIR}/preview_{SEQ}.png"))
    !python scripts/extract_ndt.py --kitti_root {KITTI} --seq {SEQ} --out_dir {NDT} --variant {VARIANT} {ndt_args}
    !tar -cf {CELLS_TAR} -C {NDT} {SEQ} {SEQ}_stats.csv config.json
    !cp {NDT}/{SEQ}_stats.csv {VDIR}/stats_{SEQ}.csv
    if os.path.isdir(f"{NDT}/{SEQ}_previews"):   # pictures of the built-in checks
        !mkdir -p {VDIR}/previews_{SEQ} && cp {NDT}/{SEQ}_previews/*.png {VDIR}/previews_{SEQ}/
    !cp {NDT}/config.json {VDIR}/config.json
if os.path.isdir(f"{NDT}/{SEQ}"):
    print(len(os.listdir(f"{NDT}/{SEQ}")), "cells files of sequence", SEQ)

# point-rotated copies of the training frames (TRAIN_PARAMS rot_points = N)
N_ROT = TRAIN_FULL["rot_points"]
if N_ROT and MODE in ("train", "smoke"):
    ROT_SUB = ex.rot_dir_name(SEQ, N_ROT)
    ROT_TAR = f"{VDIR}/{ex.rot_archive_name(SEQ, N_ROT)}"
    if not os.path.exists(ROT_TAR):
        res = subprocess.run(["python", "scripts/fetch_from_azure.py", "--sas_url", userdata.get("AZURE_SAS"),
                              "--method", "ndt", "--variant", VARIANT, "--seqs", SEQ,
                              "--rot_copies", str(N_ROT), "--drive", DRIVE],
                             capture_output=True, text=True, cwd="/content/lpr")
        print(res.stdout, res.stderr[-2000:])
    if not os.path.exists(ROT_TAR):
        args = " ".join(f"--{k} {v}" for k, v in NDT_FULL.items() if v != ex.NDT_DEFAULTS[k])
        print(f"{ROT_SUB} is not built yet. On the Azure VM (inside tmux, in the repository folder):")
        print(f'  git pull && METHOD=ndt VARIANT={VARIANT} CELL_SEQS="" ROT_COPIES={N_ROT} ROT_SEQS={SEQ}'
              + (f' ARGS="{args}"' if args else "") + ' bash scripts/azure_build.sh "<SAS URL>"')
        raise SystemExit("then run this notebook again")
    if not os.path.isdir(f"{NDT}/{ROT_SUB}"):
        !tar -xf {ROT_TAR} -C {NDT} {ROT_SUB} {ROT_SUB}_stats.csv
    print(len(os.listdir(f"{NDT}/{ROT_SUB}")), "point-rotated cells files in", ROT_SUB)

## 5. Training tuples
Not needed: the BEVPlace++ recipe finds positives and negatives from the poses itself.

In [ ]:
print("nothing to do (positives / negatives come from the poses inside the training script)")

## 6. Smoke test (MODE = "smoke"): 30 iterations
Seconds per iteration x 750 is roughly one epoch; the log also shows the peak GPU memory.

In [ ]:
if MODE == "smoke":
    !rm -rf /content/smoke
    !python scripts/train_myndt_kitti.py {train_args} --log_dir /content/smoke --epochs 1 --max_iters 30 --eval_every 0 2>&1 | grep -v Warning
else:
    print("skipped (MODE =", MODE + ")")

## 7. Training (MODE = "train"; continues an unfinished run, skips a finished one)

In [ ]:
if MODE == "train":
    print("run:", RUN_DIR, f"({run_state})")
    !python scripts/train_myndt_kitti.py {train_args} --log_dir {RUN_DIR} 2>&1 | grep -v Warning
else:
    print("skipped (MODE =", MODE + ")")

## 8. NDT cells of the evaluation sequences
Sequences that are not on Drive yet are copied from the Azure blob container (SAS URL in Colab Secrets as
`AZURE_SAS`, permissions read + list; the cells are built on the Azure VM with `METHOD=ndt bash scripts/azure_build.sh`).

In [ ]:
need = sorted(set(ev.ALL_SEQS) if ("seqs_heading" in EVALS or "compass" in EVALS) else set()
              | ({ev.EVAL_TYPES["tnet"]["params"]["seq"]} if "tnet" in EVALS else set()))
if MODE == "smoke":
    print("skipped (MODE = smoke)")
else:
    missing = [s for s in need if not os.path.exists(f"{VDIR}/cells_{s}.tar")]
    if missing:
        res = subprocess.run(["python", "scripts/fetch_from_azure.py", "--sas_url", userdata.get("AZURE_SAS"),
                              "--method", "ndt", "--variant", VARIANT, "--seqs", *missing, "--drive", DRIVE],
                             capture_output=True, text=True, cwd="/content/lpr")
        print(res.stdout, res.stderr[-2000:])
    for s in need:
        if os.path.exists(f"{VDIR}/cells_{s}.tar") and not os.path.exists(f"{NDT}/{s}/000000.bin"):
            os.makedirs(NDT, exist_ok=True)
            subprocess.run(["tar", "-xf", f"{VDIR}/cells_{s}.tar", "-C", NDT], check=True)
    print("NDT cells on this machine for:", [s for s in need if os.path.exists(f"{NDT}/{s}/000000.bin")])

## 9. Evaluations (`EVALS`, checkpoint `EVAL_CKPT`)
Each type is stored in `<run>/eval/<type>/<checkpoint>/` and is not repeated once complete.

In [ ]:
import pandas as pd
from IPython.display import Image, display

def run_evals(method, run_dir, ckpt, data_dir):
    """Every type in EVALS on one checkpoint; results in <run_dir>/eval/<type>/<checkpoint>/."""
    for e in EVALS:
        if method not in ev.EVAL_TYPES[e]["methods"]:
            continue
        res = subprocess.run(["python", "scripts/evaluate.py", "--method", method, "--run_dir", run_dir,
                              "--ckpt", ckpt, "--eval_type", e, "--data_dir", data_dir, "--kitti_root", KITTI],
                             capture_output=True, text=True, cwd="/content/lpr")
        print(res.stdout if res.returncode == 0 else res.stdout[-2000:] + res.stderr[-3000:])
        if res.returncode == 0:
            out = ev.eval_dir(run_dir, e, ev.resolve_ckpt(run_dir, ckpt))
            for png in sorted(glob.glob(f"{out}/*.png")):
                display(Image(png))

if MODE == "smoke":
    print("skipped (MODE = smoke)")
elif not glob.glob(f"{RUN_DIR}/epoch_*.pth"):
    print("no checkpoint in", RUN_DIR)
else:
    run_evals("myndt", RUN_DIR, EVAL_CKPT, NDT)

## 10. Tables of all runs and evaluations, training curves

In [ ]:
import matplotlib.pyplot as plt
!python scripts/summarize_runs.py --drive {DRIVE} > /dev/null
display(pd.read_csv(f"{DRIVE}/runs_summary.csv", dtype={"run": str}))
if os.path.exists(f"{RUN_DIR}/metrics.csv"):
    m = pd.read_csv(f"{RUN_DIR}/metrics.csv").drop_duplicates("epoch", keep="last")
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(m.epoch, m.train_loss, marker="o"); ax[0].set_title("training loss"); ax[0].set_xlabel("epoch")
    if "R@1" in m:
        ax[1].plot(m.epoch, m["R@1"], marker="o", label="R@1"); ax[1].plot(m.epoch, m["R@1%"], marker="o", label="R@1%")
    ax[1].set_title(f"KITTI {SEQ} recall (%), monitoring only  {RUN}"); ax[1].set_xlabel("epoch"); ax[1].legend()
    plt.show()

## 11. NDT-Transformer vs BEVPlace++ (every run with a `seqs_heading` evaluation)

In [ ]:
p = f"{DRIVE}/evals_summary.csv"
if os.path.exists(p):
    e = pd.read_csv(p, dtype={"seq": str, "run": str, "checkpoint": str})
    sh = e[e.eval_type == "seqs_heading"]
    print("Recall@1 per sequence")
    display(sh.pivot_table(index=["method", "run", "checkpoint"], columns="seq", values="R@1"))
    cols = [c for c in sh.columns if c.startswith("R@1 ")]
    for s in ("02", "08"):
        print(f"KITTI {s}: Recall@1 per heading group (share of queries in the % columns)")
        g = sh[sh.seq == s].set_index(["method", "run", "checkpoint"])
        display(g[[c for c in g.columns if c.startswith("% ")] + cols])
    cp = e[e.eval_type == "compass"]
    if len(cp):
        print("Compass: Recall@1 per sequence (columns) and compass error sigma in degrees (rows)")
        display(cp.pivot_table(index=["method", "run", "checkpoint", "sigma_deg"], columns="seq", values="R@1"))
    tn = e[e.eval_type == "tnet"]
    if len(tn):
        print("T-Net check (scans rotated by 'angle')")
        display(tn.dropna(axis=1, how="all").set_index(["run", "checkpoint", "angle"])
                .drop(columns=["method", "eval_type", "complete", "seq"], errors="ignore"))